In [11]:
## 计算过程

### 1. 提取转移统计
给定字符序列 **"ababc"**，长度为 5。一阶马尔可夫转移（共 4 次）为：

- `a` → `b` （第1位到第2位）
- `b` → `a` （第2位到第3位）
- `a` → `b` （第3位到第4位）
- `b` → `c` （第4位到第5位）

统计以 **'b'** 为前一个状态（$x_{t-1}=b$）的转移次数：

| 转移目标 ($x_t$) | 实际出现次数 |
| :---: | :---: |
| `a` | 1 （来自 `b`→`a`） |
| `b` | 0 （未出现） |
| `c` | 1 （来自 `b`→`c`） |

因此，以 `b` 为前缀的**总转移次数**为：
$C(b \rightarrow \cdot) = 1 + 0 + 1 = 2$

词汇表大小 $|V| = 3$（即 `{'a', 'b', 'c'}`）。

---

### 2. 拉普拉斯平滑（加 1 平滑）公式
对于一阶马尔可夫模型，平滑后的条件概率计算公式为：

$$P(x_t \mid x_{t-1}) = \frac{C(x_{t-1} \rightarrow x_t) + 1}{C(x_{t-1} \rightarrow \cdot) + |V|}$$

其中 $C(x_{t-1} \rightarrow \cdot)$ 表示以 $x_{t-1}$ 为前驱状态的所有转移总次数。

---

### 3. 代入数值计算

#### 1) $P(\text{'a'} \mid \text{'b'})$
$$P(a \mid b) = \frac{C(b \rightarrow a) + 1}{C(b \rightarrow \cdot) + 3} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

#### 2) $P(\text{'c'} \mid \text{'b'})$
$$P(c \mid b) = \frac{C(b \rightarrow c) + 1}{C(b \rightarrow \cdot) + 3} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

---

### 4. 最终答案
- **$p(\text{'a'} \mid \text{'b'}) = \frac{2}{5} = 0.4$**
- **$p(\text{'c'} \mid \text{'b'}) = \frac{2}{5} = 0.4$**

> **补充验证**：未出现的转移 $p(\text{'b'} \mid \text{'b'})$ 平滑后为 $\frac{0+1}{2+3} = 0.2$，三项概率之和 $0.4 + 0.2 + 0.4 = 1.0$，符合概率分布归一性。

SyntaxError: invalid character '，' (U+FF0C) (2427196635.py, line 4)

In [ ]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    预处理文本并生成自回归样本。

    参数：
        text (str): 输入文本
        n (int): 滑动窗口长度（特征词数）

    返回：
        vocab (dict): 词到整数ID的映射，按词频降序排列
        (features, labels) (tuple): features为列表的列表，每个子列表为n个词；
                                     labels为对应的下一个词（字符串）
    """
    # 1. 转小写，并仅保留字母和空格
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)   # 去除标点和数字，保留小写字母和空格

    # 2. 按空格分词（多个空格自动处理）
    words = text.split()

    # 3. 构建词汇表（按出现频率降序，分配ID从0开始）
    word_counts = Counter(words)
    # most_common() 返回按频率降序的列表，相同频率按首次出现顺序
    vocab = {word: idx for idx, (word, _) in enumerate(word_counts.most_common())}

    # 4. 生成滑动窗口特征和标签（忽略无后续词的窗口）
    features = []
    labels = []
    # 只有当 i+n < len(words) 时才有下一个词作为标签
    for i in range(len(words) - n):
        features.append(words[i:i+n])   # 长度为 n 的词列表
        labels.append(words[i+n])       # 下一个词

    return vocab, (features, labels)


# 示例测试（与题目描述对照）
if __name__ == "__main__":
    text = "The time machine"
    n = 2
    vocab, (feat, lab) = preprocess_text(text, n)
    print("词汇表:", vocab)          # 例如 {'the':0, 'time':1, 'machine':2} (顺序可能因频率相同而不同)
    print("特征:", feat)            # [['the', 'time']]
    print("标签:", lab)             # ['machine']

词汇表: {'the': 0, 'time': 1, 'machine': 2}
特征: [['the', 'time']]
标签: ['machine']


In [ ]:
## 线性 RNN 的梯度推导（BPTT）

设无偏置线性 RNN 定义为：

$$
h_t = W_{hh} h_{t-1} + W_{hx} x_t, \qquad o_t = W_{oh} h_t
$$

损失函数为平方损失（对所有时间步求和）：

$$
L = \frac12 \sum_{t=1}^T (o_t - y_t)^2
$$

目标：推导 $\frac{\partial L}{\partial W_{hh}}$ 的表达式，并讨论梯度消失/爆炸的条件。

---

### 1. 定义误差传播变量

令输出误差为：

$$
e_t = \frac{\partial L}{\partial o_t} = o_t - y_t
$$

定义隐状态梯度（即损失对 $h_t$ 的偏导）：

$$
\delta_t = \frac{\partial L}{\partial h_t}
$$

由链式法则，得到反向递推关系：

- 对于最后一个时间步 $T$（无后续隐状态）：

$$
\delta_T = W_{oh}^T e_T
$$

- 对于 $t < T$，考虑到 $h_t$ 同时影响 $o_t$ 和 $h_{t+1}$（通过 $W_{hh}$），有：

$$
\delta_t = W_{oh}^T e_t + W_{hh}^T \delta_{t+1}
$$

---

### 2. 对 $W_{hh}$ 的梯度展开

我们需要计算 $\frac{\partial L}{\partial W_{hh}}$。因为 $W_{hh}$ 出现在所有时间步的隐状态更新中，梯度是所有时间步贡献之和：

$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \delta_t \cdot \frac{\partial h_t}{\partial W_{hh}}
$$

关键在于计算 $\frac{\partial h_t}{\partial W_{hh}}$。由于 $h_t$ 递归依赖于 $h_{t-1}$，而 $h_{t-1}$ 也依赖于 $W_{hh}$，需要展开递归：

由 $h_t = W_{hh} h_{t-1} + W_{hx} x_t$，对 $W_{hh}$ 求导（设初始 $h_0$ 不依赖于 $W_{hh}$）：

$$
\frac{\partial h_t}{\partial W_{hh}} = h_{t-1} + W_{hh} \frac{\partial h_{t-1}}{\partial W_{hh}}
$$

递归展开至初始状态，可得：

$$
\frac{\partial h_t}{\partial W_{hh}} = \sum_{i=1}^t W_{hh}^{\,t-i} \, h_{i-1}
$$

其中 $W_{hh}^{\,t-i}$ 表示矩阵的 $(t-i)$ 次幂（标量情形即为幂次）。

因此梯度表达式为：

$$
\boxed{
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \delta_t \sum_{i=1}^t W_{hh}^{\,t-i} \, h_{i-1}
}
$$

等效地，可以交换求和顺序，写成：

$$
\frac{\partial L}{\partial W_{hh}} = \sum_{i=1}^T \left( \sum_{t=i}^T \delta_t \, W_{hh}^{\,t-i} \right) h_{i-1}
$$

---

### 3. 梯度消失与爆炸的条件

从上述表达式中，每一项都包含 $W_{hh}$ 的幂次 $W_{hh}^{\,t-i}$（或乘积 $\prod_{k=i+1}^t W_{hh}$）。这些幂次项决定了长距离依赖的梯度大小。

- **梯度爆炸条件**：当 $W_{hh}$ 的**谱半径**（即最大特征值的绝对值）大于 1 时，随着 $t-i$ 增大，$W_{hh}^{\,t-i}$ 的范数指数增长，导致梯度数值急剧增大，引发梯度爆炸。

- **梯度消失条件**：当 $W_{hh}$ 的**谱半径**小于 1 时，$W_{hh}^{\,t-i}$ 指数衰减，远距离时间步的梯度贡献趋近于 0，使得网络难以学习长期依赖，即梯度消失。

对于标量情形，条件简化为：

- $|W_{hh}| > 1$ → 梯度爆炸
- $|W_{hh}| < 1$ → 梯度消失
- $|W_{hh}| = 1$ → 梯度既不爆炸也不消失（理想边界）。

In [ ]:
import numpy as np

def rnn_step_forward(x, h_prev, W_hx, W_hh, b_h):
    """
    线性RNN单元的前向传播（tanh激活函数）。

    参数：
        x : numpy.ndarray，形状 (batch_size, input_size)
        h_prev : numpy.ndarray，形状 (batch_size, hidden_size)
        W_hx : numpy.ndarray，形状 (input_size, hidden_size)
        W_hh : numpy.ndarray，形状 (hidden_size, hidden_size)
        b_h : numpy.ndarray，形状 (hidden_size,)

    返回：
        h_next : numpy.ndarray，形状 (batch_size, hidden_size)
        cache : tuple，保存反向传播所需的中间变量
    """
    # 线性组合
    a = x @ W_hx + h_prev @ W_hh + b_h   # (N, H)
    # 激活函数
    h_next = np.tanh(a)                  # (N, H)

    cache = (x, h_prev, W_hx, W_hh, b_h, a)
    return h_next, cache


def rnn_step_backward(dh_next, cache):
    """
    RNN单元的反向传播（单步）。

    参数：
        dh_next : numpy.ndarray，形状 (batch_size, hidden_size)
                 即上游梯度 ∂L/∂h_next
        cache : tuple，来自前向传播的缓存

    返回：
        dx : numpy.ndarray，形状 (batch_size, input_size)
        dh_prev : numpy.ndarray，形状 (batch_size, hidden_size)
        dW_hx : numpy.ndarray，形状 (input_size, hidden_size)
        dW_hh : numpy.ndarray，形状 (hidden_size, hidden_size)
        db_h : numpy.ndarray，形状 (hidden_size,)
    """
    x, h_prev, W_hx, W_hh, b_h, a = cache

    # tanh 的导数：dtanh/da = 1 - tanh^2(a)
    da = dh_next * (1 - np.tanh(a) ** 2)   # (N, H)

    # 梯度链式法则
    dx = da @ W_hx.T                      # (N, D)
    dh_prev = da @ W_hh.T                 # (N, H)

    dW_hx = x.T @ da                      # (D, H)
    dW_hh = h_prev.T @ da                 # (H, H)
    db_h = np.sum(da, axis=0)             # (H,)

    return dx, dh_prev, dW_hx, dW_hh, db_h


# ------------------- 使用示例与形状验证 -------------------
if __name__ == "__main__":
    np.random.seed(0)

    # 设置超参数
    batch_size = 4
    input_size = 3
    hidden_size = 5

    # 随机生成输入数据
    x = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(hidden_size)

    # 前向传播
    h_next, cache = rnn_step_forward(x, h_prev, W_hx, W_hh, b_h)

    # 模拟上游梯度（假设损失对 h_next 的梯度为随机值）
    dh_next = np.random.randn(batch_size, hidden_size)

    # 反向传播
    dx, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)

    # 打印形状以验证
    print("梯度形状检查:")
    print(f"dx:          {dx.shape}   (期望: ({batch_size}, {input_size}))")
    print(f"dh_prev:     {dh_prev.shape} (期望: ({batch_size}, {hidden_size}))")
    print(f"dW_hx:       {dW_hx.shape}   (期望: ({input_size}, {hidden_size}))")
    print(f"dW_hh:       {dW_hh.shape}   (期望: ({hidden_size}, {hidden_size}))")
    print(f"db_h:        {db_h.shape}    (期望: ({hidden_size},))")

梯度形状检查:
dx:          (4, 3)   (期望: (4, 3))
dh_prev:     (4, 5) (期望: (4, 5))
dW_hx:       (3, 5)   (期望: (3, 5))
dW_hh:       (5, 5)   (期望: (5, 5))
db_h:        (5,)    (期望: (5,))


In [ ]:
## 深度双向 RNN 参数总数计算

### 模型设定
- **层数**：$L$（每层均为双向）
- **每层隐藏单元数**：$H$（每个方向，前向和后向各 $H$ 维）
- **输入维度**：$D$（仅第一层输入）
- **输出维度**：$O$（最终输出层）
- **忽略**：嵌入层、输出层之前的额外投影层

---

### 1. 每层 RNN 单元参数（含偏置）

对于双向 RNN，每一层包含**两个方向**（前向和后向），每个方向均有独立的权重和偏置：

- **输入到隐藏**：$W_{hx}$，形状 $(\text{input\_dim}, H)$
- **隐藏到隐藏**：$W_{hh}$，形状 $(H, H)$
- **偏置**：$b_h$，形状 $(H)$

每个方向的参数数量为：
$$
\text{per-direction} = \text{input\_dim} \times H + H \times H + H
$$

该层两个方向总计为：
$$
\text{per-layer} = 2 \times (\text{input\_dim} \times H + H^2 + H)
$$

---

### 2. 各层输入维度

- **第 1 层**：输入维度为 $D$（原始输入），参数数为：
  $$
  P_1 = 2(DH + H^2 + H)
  $$

- **第 $l$ 层（$l \ge 2$）**：输入来自上一层的双向输出拼接，维度为 $2H$。参数数为：
  $$
  P_{l} = 2\big((2H)H + H^2 + H\big) = 2(3H^2 + H) = 6H^2 + 2H
  $$

共有 $L-1$ 层这样的层，因此总 RNN 参数为：
$$
P_{\text{RNN}} = P_1 + (L-1) \times P_{l} 
= 2(DH + H^2 + H) + (L-1)(6H^2 + 2H)
$$

---

### 3. 输出层参数

最后一层的双向输出拼接为 $2H$ 维，映射到输出维度 $O$：

- 权重 $W_{oh}$：形状 $(2H, O)$，参数 $2H \cdot O$
- 偏置 $b_o$：形状 $(O)$，参数 $O$

输出层参数数：
$$
P_{\text{out}} = 2H \cdot O + O = O(2H + 1)
$$

---

### 4. 总参数表达式

$$
\boxed{
\begin{aligned}
P_{\text{total}} &= P_{\text{RNN}} + P_{\text{out}} \\
&= 2(DH + H^2 + H) + (L-1)(6H^2 + 2H) + O(2H + 1)
\end{aligned}
}
$$

**等价展开形式**：
$$
P_{\text{total}} = 2DH + (6L - 4)H^2 + 2L H + 2H O + O
$$

其中 $L \ge 1$，当 $L=1$ 时退化为单层双向 RNN。

In [ ]:
import numpy as np

class BiRNNEncoderManual:
    """
    双向 RNN 编码器（纯 NumPy 实现，无反向传播）。
    
    参数：
        input_dim  : 输入维度
        hidden_dim : 每方向隐藏单元数
        num_layers : RNN 层数（每层均为双向）
    
    方法：
        forward(X) -> outputs, final_state
            X : (seq_len, batch, input_dim)
            outputs : (seq_len, batch, 2 * hidden_dim)  每个时间步拼接状态
            final_state : (batch, 2 * hidden_dim)       最后一个时间步拼接状态
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 为每一层的前向和后向分别初始化参数（随机小值）
        self.params = []
        for l in range(num_layers):
            current_input_dim = input_dim if l == 0 else 2 * hidden_dim
            layer_params = {
                'forward': {
                    'W_hx': np.random.randn(current_input_dim, hidden_dim) * 0.01,
                    'W_hh': np.random.randn(hidden_dim, hidden_dim) * 0.01,
                    'b_h': np.zeros(hidden_dim)
                },
                'backward': {
                    'W_hx': np.random.randn(current_input_dim, hidden_dim) * 0.01,
                    'W_hh': np.random.randn(hidden_dim, hidden_dim) * 0.01,
                    'b_h': np.zeros(hidden_dim)
                }
            }
            self.params.append(layer_params)

    def forward(self, X):
        """
        X : (seq_len, batch, input_dim)
        """
        seq_len, batch, _ = X.shape
        hidden_dim = self.hidden_dim
        
        # 当前层的输入（第一层为原始 X，之后为上一层输出）
        layer_input = X
        
        for l in range(self.num_layers):
            fwd = self.params[l]['forward']
            bwd = self.params[l]['backward']
            
            # ---------- 前向 ----------
            h_f = np.zeros((seq_len, batch, hidden_dim))
            h_prev = np.zeros((batch, hidden_dim))
            for t in range(seq_len):
                x_t = layer_input[t]  # (batch, current_input_dim)
                a = x_t @ fwd['W_hx'] + h_prev @ fwd['W_hh'] + fwd['b_h']
                h_prev = np.tanh(a)
                h_f[t] = h_prev
            
            # ---------- 后向 ----------
            h_b = np.zeros((seq_len, batch, hidden_dim))
            h_next = np.zeros((batch, hidden_dim))
            for t in range(seq_len - 1, -1, -1):
                x_t = layer_input[t]
                a = x_t @ bwd['W_hx'] + h_next @ bwd['W_hh'] + bwd['b_h']
                h_next = np.tanh(a)
                h_b[t] = h_next
            
            # 拼接前向和后向，作为下一层输入
            layer_input = np.concatenate([h_f, h_b], axis=2)  # (seq_len, batch, 2*hidden_dim)
        
        outputs = layer_input
        final_state = outputs[-1, :, :]  # (batch, 2*hidden_dim)
        return outputs, final_state


# ------------------- 使用示例 -------------------
if __name__ == "__main__":
    seq_len, batch, input_dim = 10, 4, 8
    hidden_dim = 16
    num_layers = 2

    encoder = BiRNNEncoderManual(input_dim, hidden_dim, num_layers)
    X = np.random.randn(seq_len, batch, input_dim)

    outputs, final_state = encoder.forward(X)
    print("outputs 形状:", outputs.shape)        # (10, 4, 32)
    print("final_state 形状:", final_state.shape) # (4, 32)

outputs 形状: (10, 4, 32)
final_state 形状: (4, 32)


In [ ]:

- 中心词 $w_c$，其词向量为 $v_c$（输入向量，对应中心词）。
- 上下文词 $w_o$，其词向量为 $u_o$（输出向量，对应上下文词）。
- 负样本词 $w_{neg}$，其词向量为 $u_{neg}$，从噪声分布 $P_n(w)$ 中采样 $K$ 个。

在原始 Skip-gram 中，我们希望最大化给定中心词时上下文词出现的概率：
$$
P(w_o \mid w_c) = \frac{\exp(u_o^\top v_c)}{\sum_{w' \in V} \exp(u_{w'}^\top v_c)}
$$
但 softmax 分母涉及整个词表 $V$，计算量巨大。负采样将问题转化为**二分类**（区分真实上下文词和噪声词）。

---

### 2. 负采样的目标函数（对数似然）

对于每一对（中心词 $w_c$，上下文词 $w_o$），我们构造正样本 $(w_c, w_o, 1)$ 和 $K$ 个负样本 $(w_c, w_{neg}, 0)$，其中负样本词 $w_{neg} \sim P_n(w)$。

使用逻辑回归（sigmoid）建模条件概率：
- 正样本的似然：$P(\text{正} \mid w_c, w_o) = \sigma(u_o^\top v_c)$，其中 $\sigma(x) = 1/(1+e^{-x})$。
- 负样本的似然：$P(\text{负} \mid w_c, w_{neg}) = 1 - \sigma(u_{neg}^\top v_c) = \sigma(-u_{neg}^\top v_c)$。

取对数似然，对于单个中心词 $w_c$ 和一个上下文词 $w_o$，目标函数（要最大化）为：
$$
\mathcal{L}(w_c, w_o) = \log \sigma(u_o^\top v_c) + \sum_{k=1}^{K} \mathbb{E}_{w_k \sim P_n} \big[ \log \sigma(-u_{k}^\top v_c) \big]
$$

实际实现中，对每个训练样本，我们独立地从 $P_n$ 中采样 $K$ 个负样本词 $\{w_1, \dots, w_K\}$，则其经验损失（负对数似然，需最小化）为：

$$
\boxed{
\mathcal{L}_{\text{NEG}} = - \log \sigma(u_o^\top v_c) - \sum_{k=1}^{K} \log \sigma(-u_k^\top v_c)
}
$$

等价地，可写为：
$$
\mathcal{L}_{\text{NEG}} = - \log \sigma(u_o^\top v_c) - \sum_{k=1}^{K} \log \big(1 - \sigma(u_k^\top v_c)\big)
$$

---

### 3. 从噪声分布中采样负样本

噪声分布 $P_n(w)$ 通常选择为词频的幂次分布，常见设置：
$$
P_n(w) = \frac{\text{count}(w)^\alpha}{\sum_{w'} \text{count}(w')^\alpha}
$$
其中 $\alpha$ 常取 $0.75$ 或 $0.5$（Mikolov 等人建议 $\alpha=0.75$，可平滑高频词和低频词的采样概率）。

采样过程：
- 预先计算所有词的无偏经验分布（基于语料词频），并取 $\alpha$ 次幂，归一化。
- 在训练时，对每个正样本（中心词，上下文词），独立地从这个分布中抽取 $K$ 个词作为负样本（注意避免恰好抽到当前上下文词本身，但实际操作中通常允许，因为概率很小）。

---

### 4. 完整目标函数（对整个语料）

对整个语料（所有中心词及其上下文词窗口），总损失为所有样本损失之和：

$$
\boxed{
\mathcal{J} = - \sum_{(w_c, w_o) \in \mathcal{D}} \left[ \log \sigma(u_o^\top v_c) + \sum_{k=1}^{K} \log \sigma(-u_k^\top v_c) \right]
}
$$

其中 $\mathcal{D}$ 是所有（中心词，上下文词）对的集合，每个负样本 $w_k$ 独立从 $P_n$ 采样。

---

### 5. 补充说明
- 负采样将多分类问题转为二分类，极大降低了计算复杂度。
- 梯度更新时，仅更新中心词向量 $v_c$、正样本上下文词向量 $u_o$ 以及 $K$ 个负样本词向量 $u_k$，其他词向量不变。
- 此目标函数是**最大化**正样本概率同时**最小化**负样本概率的等价形式。

In [ ]:
import numpy as np

def cbow_forward(context_indices, target_indices, W, W_out):
    """
    CBOW 模型前向传播（完整 Softmax，无负采样）。

    参数：
        context_indices : numpy.ndarray，形状 (batch_size, context_size)
                          每个样本的上下文词索引（整数）
        target_indices  : numpy.ndarray，形状 (batch_size,)
                          每个样本的中心词索引（整数）
        W               : numpy.ndarray，形状 (V, d)
                          输入嵌入矩阵（词向量）
        W_out           : numpy.ndarray，形状 (d, V)
                          输出权重矩阵

    返回：
        loss : float，批量样本的平均交叉熵损失
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape

    # 1. 获取上下文词嵌入并求平均 -> hidden layer
    # context_indices: (batch, ctx) -> 取 W 中对应行，得到 (batch, ctx, d)
    emb_context = W[context_indices]          # (batch, ctx, d)
    hidden = np.mean(emb_context, axis=1)     # (batch, d)

    # 2. 计算输出 logits
    logits = hidden @ W_out                   # (batch, V)

    # 3. 数值稳定的 Softmax
    logits_max = np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits - logits_max)  # (batch, V)
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)  # (batch, V)

    # 4. 交叉熵损失（平均）
    # 选择目标词的预测概率
    batch_indices = np.arange(batch_size)
    target_probs = probs[batch_indices, target_indices]   # (batch,)
    loss = -np.mean(np.log(target_probs + 1e-12))        # 添加极小值防止 log(0)

    return loss


# ------------------- 使用示例 -------------------
if __name__ == "__main__":
    np.random.seed(42)

    # 超参数
    V = 10        # 词汇表大小
    d = 5         # 嵌入维度
    batch_size = 4
    context_size = 2

    # 随机初始化权重
    W = np.random.randn(V, d) * 0.01
    W_out = np.random.randn(d, V) * 0.01

    # 生成随机样本
    context_indices = np.random.randint(0, V, size=(batch_size, context_size))
    target_indices = np.random.randint(0, V, size=(batch_size,))

    loss = cbow_forward(context_indices, target_indices, W, W_out)
    print(f"平均交叉熵损失: {loss:.4f}")

平均交叉熵损失: 2.3025


In [ ]:
## 缩放点积注意力计算示例

给定：
- 查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$
- 键矩阵 $K \in \mathbb{R}^{3 \times 4}$
- 值矩阵 $V \in \mathbb{R}^{3 \times 5}$
- 维度 $d_k = 4$，缩放因子 $\frac{1}{\sqrt{d_k}} = \frac{1}{2}$

假设具体数值如下（为方便演示）：

$$
Q = \begin{bmatrix}
1 & 2 & 3 & 4 \\
5 & 6 & 7 & 8
\end{bmatrix}, \quad
K = \begin{bmatrix}
9 & 8 & 7 & 6 \\
5 & 4 & 3 & 2 \\
1 & 2 & 3 & 4
\end{bmatrix}, \quad
V = \begin{bmatrix}
1 & 2 & 3 & 4 & 5 \\
6 & 7 & 8 & 9 & 10 \\
11 & 12 & 13 & 14 & 15
\end{bmatrix}
$$

---

### 步骤 1：计算得分矩阵（缩放点积）

得分矩阵 $S = \frac{Q K^T}{\sqrt{d_k}} = \frac{1}{2} Q K^T$

首先计算 $Q K^T$（$2 \times 3$）：

- 第 1 行第 1 列：$1\times9 + 2\times8 + 3\times7 + 4\times6 = 9+16+21+24 = 70$
- 第 1 行第 2 列：$1\times5 + 2\times4 + 3\times3 + 4\times2 = 5+8+9+8 = 30$
- 第 1 行第 3 列：$1\times1 + 2\times2 + 3\times3 + 4\times4 = 1+4+9+16 = 30$
- 第 2 行第 1 列：$5\times9 + 6\times8 + 7\times7 + 8\times6 = 45+48+49+48 = 190$
- 第 2 行第 2 列：$5\times5 + 6\times4 + 7\times3 + 8\times2 = 25+24+21+16 = 86$
- 第 2 行第 3 列：$5\times1 + 6\times2 + 7\times3 + 8\times4 = 5+12+21+32 = 70$

所以：
$$
Q K^T = \begin{bmatrix}
70 & 30 & 30 \\
190 & 86 & 70
\end{bmatrix}
$$

缩放（除以 2）：
$$
S = \frac{1}{2} \begin{bmatrix}
70 & 30 & 30 \\
190 & 86 & 70
\end{bmatrix}
= \begin{bmatrix}
35 & 15 & 15 \\
95 & 43 & 35
\end{bmatrix}
$$

---

### 步骤 2：对得分矩阵每行应用 Softmax

对 $S$ 的每一行计算 softmax（指数化后归一化）：

**第 1 行**（$[35, 15, 15]$）：
- 指数：$e^{35}, e^{15}, e^{15}$
- 由于数值极大，为简洁保留符号，实际计算时可令 $e^{35}$ 为主导。
- 归一化：
  $$
  \alpha_{1,1} = \frac{e^{35}}{e^{35}+e^{15}+e^{15}},\quad
  \alpha_{1,2} = \frac{e^{15}}{e^{35}+e^{15}+e^{15}},\quad
  \alpha_{1,3} = \frac{e^{15}}{e^{35}+e^{15}+e^{15}}
  $$
  显然，$\alpha_{1,1} \approx 1$，$\alpha_{1,2}, \alpha_{1,3} \approx 0$（指数尺度差异巨大）。

**第 2 行**（$[95, 43, 35]$）：
- 同理，$\alpha_{2,1} = \frac{e^{95}}{e^{95}+e^{43}+e^{35}} \approx 1$，其余近似为 0。

为了精确起见，我们保留 softmax 的精确表达式，但注意数值上第一列几乎为 1。

---

### 步骤 3：加权求和得到输出矩阵

输出矩阵 $O = A \cdot V$，其中 $A$ 是 softmax 权重矩阵（$2 \times 3$），$V$ 为 $3 \times 5$。

令：
$$
A = \begin{bmatrix}
a_{11} & a_{12} & a_{13} \\
a_{21} & a_{22} & a_{23}
\end{bmatrix}
$$
其中 $a_{ij} = \frac{\exp(S_{ij})}{\sum_k \exp(S_{ik})}$。

那么输出 $O$ 的第 $i$ 行为 $A$ 的第 $i$ 行与 $V$ 的加权和：
$$
O_{i,*} = \sum_{j=1}^{3} a_{ij} \cdot V_{j,*}
$$

**第 1 行输出**：
$$
O_{1,*} = a_{11} \cdot [1,2,3,4,5] + a_{12} \cdot [6,7,8,9,10] + a_{13} \cdot [11,12,13,14,15]
$$

**第 2 行输出**：
$$
O_{2,*} = a_{21} \cdot [1,2,3,4,5] + a_{22} \cdot [6,7,8,9,10] + a_{23} \cdot [11,12,13,14,15]
$$

---

### 最终输出矩阵

由于 $S$ 的第一列远大于其余列，$A$ 近似为：
$$
A \approx \begin{bmatrix}
1 & 0 & 0 \\
1 & 0 & 0
\end{bmatrix}
$$
因此输出近似等于 $V$ 的第一行（两行相同）：
$$
O \approx \begin{bmatrix}
1 & 2 & 3 & 4 & 5 \\
1 & 2 & 3 & 4 & 5
\end{bmatrix}
$$

若保留精确 softmax 计算，可代入具体数值（但数值差异极大，结果几乎相同）。

---

### 一般公式总结

对于任意矩阵，计算步骤为：
1. $S = \frac{Q K^T}{\sqrt{d_k}}$，形状 $(2,3)$。
2. $A = \text{softmax}(S)$（按行），形状 $(2,3)$。
3. $O = A V$，形状 $(2,5)$。

此即缩放点积注意力的输出矩阵。

In [ ]:
import numpy as np

class MultiHeadAttentionNumpy:
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # = 2

        # 初始化权重矩阵 (无偏置)，使用小随机数
        # 形状均为 (d_model, d_model)，线性变换为 x @ W
        self.W_q = np.random.randn(d_model, d_model) * 0.01
        self.W_k = np.random.randn(d_model, d_model) * 0.01
        self.W_v = np.random.randn(d_model, d_model) * 0.01
        self.W_o = np.random.randn(d_model, d_model) * 0.01

    def softmax(self, x, axis=-1):
        """数值稳定的 softmax"""
        e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
        return e_x / np.sum(e_x, axis=axis, keepdims=True)

    def forward(self, X):
        """
        X : (seq_len, batch, d_model)
        返回 : (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape

        # 1. 线性投影 (无偏置)
        Q = X @ self.W_q   # (seq_len, batch, d_model)
        K = X @ self.W_k
        V = X @ self.W_v

        # 2. 重塑为多头形式: (seq_len, batch, num_heads, d_k)
        Q = Q.reshape(seq_len, batch, self.num_heads, self.d_k)
        K = K.reshape(seq_len, batch, self.num_heads, self.d_k)
        V = V.reshape(seq_len, batch, self.num_heads, self.d_k)

        # 3. 交换维度以便批量矩阵乘法: (batch, num_heads, seq_len, d_k)
        Q = np.transpose(Q, (1, 2, 0, 3))  # (batch, num_heads, seq_len, d_k)
        K = np.transpose(K, (1, 2, 0, 3))
        V = np.transpose(V, (1, 2, 0, 3))

        # 4. 缩放点积注意力
        scores = np.matmul(Q, np.transpose(K, (0, 1, 3, 2))) / np.sqrt(self.d_k)
        # scores: (batch, num_heads, seq_len, seq_len)

        attn_weights = self.softmax(scores, axis=-1)  # (batch, num_heads, seq_len, seq_len)

        attn_output = np.matmul(attn_weights, V)      # (batch, num_heads, seq_len, d_k)

        # 5. 合并多头：转置回 (seq_len, batch, num_heads, d_k) 并合并
        attn_output = np.transpose(attn_output, (2, 0, 1, 3))  # (seq_len, batch, num_heads, d_k)
        attn_output = attn_output.reshape(seq_len, batch, self.d_model)  # (seq_len, batch, d_model)

        # 6. 最终线性层
        out = attn_output @ self.W_o   # (seq_len, batch, d_model)

        return out


# ------------------- 使用示例 -------------------
if __name__ == "__main__":
    d_model = 4
    num_heads = 2
    seq_len = 3
    batch = 2

    # 创建模型
    mha = MultiHeadAttentionNumpy(d_model, num_heads)

    # 随机输入
    X = np.random.randn(seq_len, batch, d_model)

    # 前向传播
    output = mha.forward(X)

    print("输入形状:", X.shape)        # (3, 2, 4)
    print("输出形状:", output.shape)   # (3, 2, 4)

输入形状: (3, 2, 4)
输出形状: (3, 2, 4)
